# E11 — 30-Problem Mechanism, Harm, and Boundary Ablation

**Independent mechanism/generalization experiment, strictly calibration-only.** 30 previously unused problems (9 Arithmetic, 11 Countdown, 10 Multistep) × one fixed source + four actions = **150 new model calls**. This measures (i) repair recovery, (ii) damage to initially correct answers, (iii) answer-format finalization, (iv) the location heuristic against a fixed-tail and global control, and (v) actual inference costs. There is no overlap with E10's 36-test manifest.

The E06 policy remains locked; **E11 results must not tune the E06 router**. Saves immediately to `MyDrive/RepairScopeX/E11_Mechanism_V1/records`. Run All executes up to 12 new calls or 29 minutes; repeat the same notebook to resume; never repeat completed IDs. GPU runtime has not been executed here.

In [ ]:

import os,sys,subprocess
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
CODE_COMMIT='fe225a99e0b5aeff351804e871d7581155d7f27d'
REPO_DIR='/content/ALoDLM_RSX_E10E11'
if not os.path.isdir(os.path.join(REPO_DIR,'.git')):
    subprocess.run(['git','clone','-q','https://github.com/amazon-science/ALoDLM.git',REPO_DIR],check=True)
subprocess.run(['git','-C',REPO_DIR,'checkout','--detach','-q',CODE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','HEAD'],text=True).strip()==CODE_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.56.1','accelerate==1.10.1','huggingface-hub>=0.34,<1.0','safetensors>=0.6','PyYAML>=6'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',REPO_DIR,'--no-deps'],check=True)
print('Pinned ALoDLM source:',CODE_COMMIT)


In [ ]:
import json,os,re,time,random,math,ast,hashlib,zipfile,zlib,base64,gc,csv
from pathlib import Path
from collections import Counter,defaultdict
from types import SimpleNamespace
import numpy as np,torch,pandas as pd
from huggingface_hub import snapshot_download
from google.colab import drive,files
from IPython.display import display
if not torch.cuda.is_available(): raise RuntimeError('GPU runtime required (T4 or better)')
if torch.cuda.get_device_properties(0).total_memory<12*2**30: raise RuntimeError('GPU must have >=12GiB VRAM')
drive.mount('/content/drive')
STUDY='RSX_E11_Mechanism_v1'
ROOT=Path('/content/drive/MyDrive/RepairScopeX/E11_Mechanism_V1')
RECS=ROOT/'records';RECS.mkdir(parents=True,exist_ok=True)
DATA=json.loads(zlib.decompress(base64.b64decode('eNrFW11v2zgW/SuCn92C5OVnsNhFH7rYAQbzMLOLXWAx8FySl41QxcpKctPuYP77XNpJ6qSJa3UcD5APR6JE6h6ec88llV8XCUcaFxf//XWR+q5rx7Zfrwaahk+LC7FcvG/XeXGxwKGdLq9oatNiuVhvriIN9Zqfl4vroY8dXa1Ku35Hw/XQriduX8jFIoUCV5zwShVNBbLOBtBk6ZMRyVjSwRhFIXhpfbYpOimdwhBzWXy+cVv7H8aPKlwk7No44MRDvPg8ogshnN9dcHVdO//nJQ3UIH+r0LSZ1twIuyb2H2lcNoTpskn9esJ2zWNupGmuaT2+bv7R3zRXuP60/XN7+bS7UTf176h+/lvzdp2bG+64+ft3P7z5fvXmh5/+/fbHi+Yvu5D89TUPY7zu2jqKvcHy0QkHvsfiQoN5zWGd6Oq6w4kWF+q35Ukjj9mCskLLZJI2ARCISIpslE45erBaCWGTRGII0CatlI3gfS4WdBI4M/Ie9iP/ptmNsOlLM3KD9zzYph2b3H5gHHJD/9tg131q8KqvkZdNumy7PNB62SBHlj7Q8Gl3rBkoUfuBxsbY+1vtYXR/9xu6B6of2nftunZwAqCstA+BghMDVXKgiEKWBJQy00BCESI4X7IQCZOq4GAUOYJWXqCOAC4o1FL5ZGOcCZR5BNR4SV1pLvsuj43XTI7+Pcf37UdMEwPEpNge2dIg9sPQ31DeC//u5EBXTKJTBDs8jLU8cay91kUzMxKwvjiWJhYZK0qA7IqVCAo0CMtapUhqC0HbHFzgVmiCTtLNjLXVD2Ndn4RSbXcnPGNjLX9uq+68acrQEocQM2MBornqOei3J+8jvv17T5XW/c0JAh8eTXJx4sA7FSIHPyRQKlmdSzA+yGg4IcjIgc45OW94tjMyVrlC4LRTCazwQfgMcwMfDkxy8XiSh7POcSdfdo4nE1TOGNEFVouktSy2gE0oOfhoSBfJOTgGSNZVKqCCbET2KAz67PVcPTHfLvw8x48Qfi3+FOGXnB9fVvmZBYhaW4Iidaq67q0NyQErjnCWZan4aAPJRDkV9IqpAEJh5ks8ZTHXHP0BpMwxQFn4U4ACaV4WJx1skaqIVJSQQpLynDqA7SwnBSFKNo7lKrNq8WnkNKKdzkoozdZKo9FzxcuLr2eNoJ/OGhLOmTWkcC+bNkzxWZLnVOAox4S6CJ+0IC2cS8pIGVKEqNg3OXZJikxgnigAwwCoLGbna/l82tDwOG0oe9a8Id28vJH6zXrK/c36QaiVXkpYKng25FHpEkKIgqyPxBH1IgbOC8rpGJzDWLxGVME79MDJxebE5pQBipgyGv/1kN8P7IuC7V/jdn7eKtPY8GB5Ru8UR0FDt4Hv14nb9Q1flzdpd8kuSo3y4XXzI02bYd1g84F7ZWLcQ8x3uB5orPHaqdgdJr88DcovX0eldvgAlvo0+apdb8bFNwGkl26pwrPwKMtWNZJRSrssmQuhGO+yiMQ1swzBKgsGY3JJZxaqVIplKSpc8vHsLibMg8fIQ/AwOu4WnHAMOK+8Py84tcOH4Ew3baJvA0YxLpK/nq+4yTumjghcLWSB3hbDJtdy4UAMA7NEe124phBSC2vZ1IpgbdEZi3IZ8lxowkHmMDA82h048hhw7JmJ87jUW4ybq28EJiylXUrxvKSVxI7XuZCNlbEEBz5Wv6WkKwZSCMUzUlXjUvaGqz/JJbmuyYWcMhnnAWPtQWACS5rdASPFMcBIp8+LTO3wZKxh+TZL9Tw2imtvZYQVlm0tSpmyQmImRWGK0ZHJIZU1PsdgsyfhiIrzvggHSbsoYWa6cYewqanG3HLmKGheyWDOrGi1x9NJmmDasB94fr2Ec7oIkjAUlYIsZKwxnOohg6QQ0SIISop/KQRSNmpGL2Yg4/kymImOPSxpgpkjbpkDxzHn3MR5bAW6zfgHrACLOOcc+3zC4SLQMxEKZSpC0W6F0ETmVPSFlNEiJ5cxJ6MiaSdiKllzjcnZQwPYeeho87WEc2cG7FHcMRLOzJ3a48m4A2LJX9IfUDb2ZElSLkAhsSuQQjlApaQgoCRTSTzlDWIgU5XJewB0YKBoCkynmejAIXR4pNvvLXf8MfCAFedFp3Z4uqzDlsAspXsWHDCW7XMErCswLF4iMQAMRMpFiqJKKQ6Af6uYhAvsDkAHIGFUigypKvPA8QdttKyW4DbvSHdUlWPPbKRrhyetcviZeT4+L202piitSEZ6LnUM1kJUQ7IUJFrWMaFtCKZwnWNBZvZv0mEwXBJpoQWGmVWodl/D5448R0mbsWd2bLXDk8ITltVUm2fhCSA9xx4ySi5zONVz+sk2c53jrUuZUEWVDHoohr2BM7JwcSpNBm1LYSrNLHUOOmpGp5rqLXvMUezR9szs0fZb4bnadFM7TnR93HpZ1CYbAdkEqYnxYYfgs3Iuexu9YDcdAoSSFVSPYHRATkAkTclWSJQpfR2X+wF94abfNGO67PuuiZtPTBrbXGN6T9O4v8eu6556artxt/u+Y9QT7dSDdg+35LeH2zWDO2F3iv0YO29dbSYkxkcAw0VLiZxHNBVFpoBxTofgGIWUU10ldk44HX3imtQGAqFM4KbFzYTE+oeQ3OBAl/2GKRPpXV0C3obK74KO72h83Xw3fV7Bd03CYerv2tVXIm7bbYFYVhqtm/GyvR6b/Xt8xueu+emWPb2Zt8Y8Ex/rtqv1lBkMb7VUACGQK1orJ4zUDiIpsDaSKDIyuaxRVBK7bjQhBj0TH30MPs48gw88xCccgEeK8+AT1IvCAyV4xgKcYxeQsZQco/ZcgiaZQwFpyLKGZfLJKmSqOeZNXWmLBoMNBefBYw4pmnxKqeQTknYb+oPa97KaJoN+UVGLodgEUheZtJOYSsgyGzZhVLfFMICSZCApVMU74TM3ccKSiRR9jC7PJI04hjRBPEMa/ZA0Sh9gjTuTqLnwoqxRhoRSOetEEa3PpngmjfDITpl8CEo5zfm+eEVkMQVMCCAIWdAyO287kzWPks47HDKtaWj44dZMAt8M/c1dSnFN6fqbatx2NPien39gjO6O1s00jnP/4cFe2t3Z0yEAat77dzMRQKh7LuisijJDLBRM3ZMh0pKTRjARpWKNIp2CkEoRn4siMDds0c4f86bRPgLeHkbA7SFgnwQAzg0AhBeNv1d+u/SlkooQ2HZlqyMmbzFH9sSS03gELhXr4lhhhQHDxYxmj4zghRJmJgO+2Djur5tL5IQt78WnL03dMJ6mjrbb9pm6dvsWxW7j3jQjP0BHn5vsbSRvj2yRwQ/YdsjjOgEESsh5L07M5QCyn2XZlwmwFn4aVNRcJeZc30vKQMp44SnKSLlQEtZyGi8hleg1mTQzd3v3DAYyHIuBPz8GX+jQiTHg2FpHhdOBFjpZyaHNDiCHmMkmEa0XhqvBrJWWaGROUmRWKRQ+sREuMBMDdViH9H4mUE8K0V6COJcSycdK9HO9hNLi4tcFbl+8qSFelHYYp9UVTpcrWk9Df/1pe5O2M2LvwLuuj9jtHcgtO5Fp1a9p1bX8Qyq/4A6w61a3N19d07C6jfLiYho2tFzETebBjXUIT93ggn8snxoRI7SNC0+C1bgppf345Zgugv1i4E9c9xtPsJ4nYGmn1Vth70Y29psh0XbpiH3rSMGuOipTVcDdqdVIt+8r8Seqk+bHn/7z6q2Ur64oXeK6Ha9efdg2nza59synVnvQrO6brbgZj4KdXl4xky/rO3MoyVqRlNKeSIaoFc/ogFaSYmGnwinUQP23goBsNqEYlnxkc5rZmgLV+by93cDOcWRqrcb2/5V10v72OwlMLdQ=')))
MODEL_ID='amazon/ALoDLM-1.7B'
MODEL_REV='588c7dc1946e3e09fd496da0b0808bd26f46a758'
Q=0.4;TAU=0.4;WINDOW=16
PRIOR_PROMPT='\n\nGive a short solution. End with one line: FINAL_ANSWER: <number>. For Countdown include an expression using all inputs once.'
DIRECT_PROMPT='\n\nDo not provide working. Reply with one line containing the final numerical answer. For Countdown provide a valid expression before the final number.'
FRESH_PROMPT='\n\nIndependently solve the problem from the beginning. Verify any computation. End with FINAL_ANSWER: <number>. For Countdown give a valid expression using each number once.'
def digest(v):
    return hashlib.sha256(json.dumps(v,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()
def save_new_json(path,obj):
    if path.exists():
        old=json.loads(path.read_text());assert old==obj,'Immutable protocol/data mismatch at '+str(path)
    else:
        temp=path.with_name(path.name+'.pending');temp.write_text(json.dumps(obj,indent=2,sort_keys=True,ensure_ascii=False));os.replace(temp,path)
def atomic_record(obj):
    assert obj['protocol_hash']==PROTOCOL_HASH
    tid=obj['trial_id'];p=RECS/(hashlib.sha256(tid.encode()).hexdigest()+'.json')
    if p.exists():assert json.loads(p.read_text())==obj,'Conflicting trial ID '+tid
    else:
        t=p.with_suffix('.pending');t.write_text(json.dumps(obj,sort_keys=True,indent=2,ensure_ascii=False));os.replace(t,p)
def existing(tid):
    p=RECS/(hashlib.sha256(tid.encode()).hexdigest()+'.json')
    if not p.exists():return None
    obj=json.loads(p.read_text());assert obj['trial_id']==tid and obj['protocol_hash']==PROTOCOL_HASH
    return obj
def read_records():
    out=[json.loads(f.read_text()) for f in sorted(RECS.glob('*.json'))]
    assert len(out)==len({r['trial_id'] for r in out})
    assert all(r['protocol_hash']==PROTOCOL_HASH for r in out)
    return out
def set_seed(seed):
    random.seed(seed);np.random.seed(seed%(2**32-1));torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
def make_prompt(content):
    return list(tokenizer.apply_chat_template([{'role':'user','content':content}],tokenize=True,add_generation_prompt=True,enable_thinking=False))
def render(ids):return tokenizer.decode(list(map(int,ids)),skip_special_tokens=True).strip()
def model_call(prompt,prefix,mode,max_new,seed):
    set_seed(seed)
    cfg=DecodeConfig(q=Q,tau=TAU,mode=mode,window_size=WINDOW,max_new_tokens=int(max_new),position_penalty=None)
    cfg.validate()
    torch.cuda.reset_peak_memory_stats();start=time.monotonic()
    out=decoder.generate(make_prompt(prompt)+list(map(int,prefix)),cfg)
    new=list(map(int,out['token_ids']))
    return {'out':render(list(prefix)+new),'ids':list(prefix)+new,'new_ids':new,
            'generated_tokens':int(out.get('generated_tokens',len(new))),
            'loop_token_passes':int(out.get('loop_token_passes',0)),
            'outer_steps':int(out.get('outer_steps',0)),
            'stop_reason':str(out.get('stop_reason','unknown')),
            'wall_seconds':round(time.monotonic()-start,3),
            'peak_vram_gb':round(float(torch.cuda.max_memory_allocated()/2**30),3)}
def seed_for(pid):return 602600 + int(hashlib.sha256(pid.encode()).hexdigest()[:6],16) % 999999

def character_cut(ids,offset):
    for i in range(1,len(ids)):
        if len(render(ids[:i]))>=offset:return max(1,i-1)
    return max(1,len(ids)//2)
def repair_cut(ids,action):
    n=len(ids)
    if n<2:return 0
    if action=='first_math':
        m=re.search(r'(?:\d+\s*[+*/-]\s*\d+)|(?:\d+\s*=)',render(ids))
        return min(n-1,max(1,character_cut(ids,m.start()) if m else n//2))
    if action=='tail50':return min(n-1,max(1,n//2))
    raise ValueError(action)
def snapshot_export():
    archive=Path('/content')/(STUDY+'_Checkpoint.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(ROOT.rglob('*')):
            if p.is_file() and not p.name.endswith('.pending'):z.write(p,p.relative_to(ROOT))
    print('Archive',archive,'size',archive.stat().st_size,'saved records',len(read_records()))
    try:files.download(str(archive))
    except Exception as exc:print('Auto-download failed; file remains at',archive,repr(exc))
    return archive


In [ ]:
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')
NUM_RE=re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?')
RUN_RE=re.compile(r'[0-9()+*×÷/\-+=\s]{3,}')
def closenum(a,b):return a is not None and b is not None and math.isclose(float(a),float(b),rel_tol=1e-9,abs_tol=1e-8)
def safe_expression(s):
    try:t=ast.parse(s,mode='eval')
    except Exception:return None
    lit=[]
    def walk(n):
        if isinstance(n,ast.Expression):return walk(n.body)
        if isinstance(n,ast.Constant) and type(n.value) in (int,float):lit.append(float(n.value));return float(n.value)
        if isinstance(n,ast.UnaryOp) and isinstance(n.op,(ast.UAdd,ast.USub)):
            a=walk(n.operand);return (-a if isinstance(n.op,ast.USub) else a)
        if isinstance(n,ast.BinOp) and isinstance(n.op,(ast.Add,ast.Sub,ast.Mult,ast.Div)):
            a,b=walk(n.left),walk(n.right)
            if isinstance(n.op,ast.Add):return a+b
            if isinstance(n.op,ast.Sub):return a-b
            if isinstance(n.op,ast.Mult):return a*b
            if abs(b)<1e-12:raise ZeroDivisionError
            return a/b
        raise ValueError('Unsupported expression')
    try:
        v=walk(t)
        if math.isfinite(v):return {'value':v,'literals':lit}
    except Exception:pass
    return None

def extract_expression(text,problem):
    cleaned=text.replace('×','*').replace('÷','/')
    for m in RUN_RE.finditer(cleaned):
        raw=m.group(0)
        # Restrict to expression strings and avoid accidentally matching a target alone.
        for part in raw.split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            result=safe_expression(ex)
            if not result:continue
            if len(result['literals'])==len(problem['numbers']) and sorted(result['literals'])==sorted(map(float,problem['numbers'])) and closenum(result['value'],problem['target']):return ex
    return None

BOX_RE=re.compile(r'\\boxed\s*\{\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+))\s*\}')
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)\b(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')

def last_explicit_number(text):
    """Only explicit answer markers, never final arbitrary numerals in unfinished reasoning.
    Conflicting explicit answer values => ambiguous, never an eligible failure."""
    text=str(text)
    entries=[]
    for pattern,kind in ((FINAL_RE,'final'),(BOX_RE,'boxed'),(ANSWER_RE,'answer_phrase')):
        for m in pattern.finditer(text):
            try:entries.append((m.start(),float(m.group(1)),kind))
            except (ValueError,OverflowError):pass
    if not entries:return None,'missing'
    unique=[]
    for _,v,_ in entries:
        if not any(closenum(v,w) for w in unique):unique.append(v)
    if len(unique)>1:return None,'conflicting_markers'
    entries.sort()
    return entries[-1][1],entries[-1][2]

def countdown_wrong_candidate(text,p):
    """Inspectable wrong *complete* expression consuming exactly the supplied numbers.
    Do not equate missing expressions with semantic error."""
    for m in RUN_RE.finditer(str(text).replace('×','*').replace('÷','/')):
        for part in m.group(0).split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            r=safe_expression(ex)
            if r and len(r['literals'])==len(p['numbers']) and sorted(r['literals'])==sorted(map(float,p['numbers'])):
                if not closenum(r['value'],p['target']):return ex
    return None

def grade_output(text,p):
    text=str(text)
    num,origin=last_explicit_number(text)
    sem_numeric=closenum(num,p['target'])
    last_nonempty=next((line.strip() for line in reversed(text.splitlines()) if line.strip()),'')
    fmt=FINAL_RE.fullmatch(last_nonempty)
    strict=bool(fmt and closenum(float(fmt.group(1)),p['target']))
    if p['kind']=='countdown':
        expr=extract_expression(text,p)
        success=expr is not None
        wrong_expr=None if success else countdown_wrong_candidate(text,p)
        explicit_wrong=(num is not None and not sem_numeric)
        # An explicitly wrong answer or an inspectable illegal computation, with no valid solution.
        is_wrong=bool(not success and origin!='conflicting_markers' and (explicit_wrong or wrong_expr is not None))
    else:
        expr=None;wrong_expr=None
        success=bool(sem_numeric)
        is_wrong=bool(num is not None and not sem_numeric and origin!='conflicting_markers')
    return {'success':bool(success),'numeric_correct':bool(sem_numeric),
            'strict_correct':bool(strict),'valid_expression':expr,
            'wrong_expression':wrong_expr,'answer_origin':origin,
            'explicit_final_number':num,'eligible_semantic_failure':bool(is_wrong),
            'failure_type':('definite_wrong' if is_wrong else
                   ('semantically_correct' if success else 'unanswered_or_ambiguous'))}

def public_accept(text,p):
    """Label-blind stopping gate: never consult word-problem gold targets."""
    if p['kind']=='countdown':
        # Target is explicitly present in the Countdown input; a valid expression may be verified.
        return extract_expression(str(text),p) is not None
    num,origin=last_explicit_number(str(text))
    return num is not None and origin!='conflicting_markers'

def features(text,p,n_tokens):
    """Target-free features; no test labels or correctness queries."""
    num,origin=last_explicit_number(text)
    L=int(n_tokens)
    return {'kind':p['kind'],'length_bucket':'short' if L<45 else ('medium' if L<80 else 'long'),
            'has_final':bool(FINAL_RE.search(str(text))), 'answer_kind':origin,
            'repetition':bool(re.search(r'(\b\w+\b)(?:\s+\1){2,}',str(text),re.I))}

def feat_bucket(f):return f['kind']+'|'+('final' if f['has_final'] else 'no_final')

# Essential semantic-failure regression checks (no GPU or network needed):
q={'kind':'arithmetic','target':14,'numbers':[]}
assert grade_output(r'We have \boxed{14}.',q)['success']
assert not grade_output(r'We have \boxed{14}.',q)['eligible_semantic_failure']
assert grade_output('FINAL_ANSWER: 13',q)['eligible_semantic_failure']
assert not grade_output('Then 14 times',q)['eligible_semantic_failure']
assert not grade_output('No answer yet',q)['eligible_semantic_failure']
assert not grade_output('FINAL_ANSWER: 13\n\\boxed{14}',q)['eligible_semantic_failure']
assert grade_output('The answer is 14',q)['success']
p={'kind':'countdown','numbers':[2,3,4],'target':14}
assert grade_output('2 + 3 * 4 = 14',p)['success']
assert not grade_output('2 + 3 * 4 = 14',p)['eligible_semantic_failure']
assert grade_output('2 + 3 + 4 = 9',p)['eligible_semantic_failure']
assert not grade_output('14',p)['eligible_semantic_failure']
assert not grade_output('2 + 4 + 4 = 14',p)['success']
print('Semantic grader regression suite PASSED: correct boxed excluded, explicit wrong admitted, truncations rejected.')

def strict_completed_correct(text,p):
    ev=grade_output(str(text).replace('\\(','').replace('\\)','').replace('\\[','').replace('\\]',''),p)
    if p['kind']!='countdown':
        v,orig=last_explicit_number(str(text))
        return bool(closenum(v,p['target']) and orig!='conflicting_markers')
    if ev['valid_expression'] is None: return False
    # Accept only complete final-answer lines, NOT incomplete math reasoning.
    for match in re.findall(r'(?im)^\s*FINAL_ANSWER\s*:\s*(.*?)\s*$',str(text)):
        raw=match.strip().replace('\\(','').replace('\\)','').replace('\\[','').replace('\\]','')
        try:
            if closenum(float(raw),p['target']):return True
        except (ValueError,TypeError):pass
        exp=safe_expression(raw)
        if exp and closenum(exp['value'],p['target']) and sorted(exp['literals'])==sorted(map(float,p['numbers'])):return True
    return False

def dual_score(text,p):
    main=grade_output(text,p)
    complete=strict_completed_correct(text,p)
    return {'primary':main,'complete_correct':bool(complete),
            'grading_disagreement':bool(main['success'])!=bool(complete),
            'class':('correct' if complete else ('definite_wrong' if main['eligible_semantic_failure'] else 'ambiguous'))}
# Scoring unit tests against prior observed failure modes.
test_cd={'kind':'countdown','target':14,'numbers':[2,3,4]}
assert strict_completed_correct(r'2 + 3 * 4 = 14\nFINAL_ANSWER: 14'.replace('\\n','\n'),test_cd)
assert not strict_completed_correct('2 + 3 * 4 = 14',test_cd)
assert not strict_completed_correct(r'2 + 3 * 4 = 14\nFINAL_ANSWER: \\('.replace('\\n','\n'),test_cd)


In [ ]:

CASE_BY_ID={p['problem_id']:p for p in DATA['cases']};SPEC=DATA['spec']
assert len(CASE_BY_ID)==30 and len({p['problem_id'] for p in DATA['cases']})==30
assert all(p['split']=='calibration' for p in CASE_BY_ID.values())
ACTIONS=['source','first_math_entropy','tail50_entropy','global_entropy','direct_one_line_128']
PLAN=[(p['problem_id'],a) for p in DATA['cases'] for a in ACTIONS]
assert len(PLAN)==150
PROTOCOL={'spec':SPEC,'source_case_ids':sorted(CASE_BY_ID),
          'code_commit':CODE_COMMIT,'model_rev':MODEL_REV,'q':Q,'tau':TAU,'window':WINDOW,'plan':PLAN}
PROTOCOL_HASH=digest(PROTOCOL)
save_new_json(ROOT/'protocol.json',{'protocol_hash':PROTOCOL_HASH,'protocol':PROTOCOL})
print('150 calls (30 fresh source + 120 intervention) planned. Saved:',len(read_records()))


In [ ]:

sys.path.insert(0,REPO_DIR)
from alodlm.generate import load_decoder
from alodlm.inference import DecodeConfig
MODEL_DIR=snapshot_download(repo_id=MODEL_ID,revision=MODEL_REV)
gc.collect();torch.cuda.empty_cache()
args=SimpleNamespace(model=MODEL_DIR,model_config=None,device='cuda',precision='fp32')
decoder=load_decoder(args);tokenizer=decoder.tokenizer
assert next(decoder.model.parameters()).dtype==torch.float32
print('Pinned FP32 ALoDLM loaded')


In [ ]:

SESSION_MAX_NEW=12;SESSION_MAX_MINUTES=29
started=time.monotonic();new=0;failures=[]
for pid,phase in PLAN:
    if new>=SESSION_MAX_NEW or time.monotonic()-started>=SESSION_MAX_MINUTES*60:break
    tid=f'{STUDY}|{pid}|{phase}'
    if existing(tid) is not None:continue
    p=CASE_BY_ID[pid];src=existing(f'{STUDY}|{pid}|source');base=seed_for(pid)+917
    try:
        if phase=='source':
            res=model_call(p['prompt']+PRIOR_PROMPT,[],mode='left1',max_new=96,seed=base)
            budget=96;mode='left1';cut=0
        else:
            assert src is not None,'Missing source for action'
            tok=src['token_ids'];cut=0
            if phase=='direct_one_line_128':
                res=model_call(p['prompt']+DIRECT_PROMPT,[],mode='left1',max_new=128,seed=base);budget=128;mode='left1'
            elif phase=='global_entropy':
                res=model_call(p['prompt']+PRIOR_PROMPT,[],mode='entropy',max_new=96,seed=base);budget=96;mode='entropy'
            else:
                mode='entropy';cut=repair_cut(tok,'first_math' if phase=='first_math_entropy' else 'tail50')
                if len(tok)<2:
                    budget=0;res={'out':src['output'],'ids':tok,'new_ids':[],'generated_tokens':0,'loop_token_passes':0,
                        'wall_seconds':0.,'peak_vram_gb':0.,'stop_reason':'non_executable_short_source'}
                else:
                    budget=max(1,len(tok)-cut)
                    res=model_call(p['prompt']+PRIOR_PROMPT,tok[:cut],mode='entropy',max_new=budget,seed=base)
                    assert res['ids'][:cut]==tok[:cut]
        rec={'trial_id':tid,'protocol_hash':PROTOCOL_HASH,'phase':phase,'action':phase,'problem_id':pid,
             'kind':p['kind'],'split':'calibration','source_trial_id':src['trial_id'] if src else None,
             'output':res['out'],'stop_reason':res['stop_reason'],'generated_tokens':res['generated_tokens'],
             'loop_token_passes':res['loop_token_passes'],'wall_seconds':res['wall_seconds'],
             'peak_vram_gb':res['peak_vram_gb'],'budget':budget,'decoder':mode,'preserved_tokens':cut,
             'score':dual_score(res['out'],p),'source_output':src['output'] if src else None,
             'seed':base}
        if phase=='source':rec['token_ids']=res['ids']
        atomic_record(rec);new+=1
        print('NEW',new,phase,pid,rec['score']['class'],flush=True)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache();failures.append({'trial_id':tid,'error':'GPU_OOM'});break
    except Exception as exc:
        failures.append({'trial_id':tid,'error':repr(exc)});break
if failures:
    (ROOT/'last_failure.json').write_text(json.dumps(failures,indent=2));print('STOPPED',failures[-1])
print('Saved:',len(read_records()),'/',len(PLAN))


In [ ]:

recs=read_records();by={(r['problem_id'],r['phase']):r for r in recs};rows=[]
for pid,p in CASE_BY_ID.items():
    src=by.get((pid,'source'))
    if src is None:continue
    for action in ACTIONS:
        r=by.get((pid,action))
        if r:
            rows.append({'problem_id':pid,'kind':p['kind'],'action':action,'strict_success':int(r['score']['complete_correct']),
                'original_correct':int(src['score']['complete_correct']), 'original_category':src['score']['class'],
                'pass_cost':r['loop_token_passes']+(0 if action=='source' else src['loop_token_passes']),
                'time_seconds':r['wall_seconds']+(0 if action=='source' else src['wall_seconds']),
                'grader_disagreement':int(r['score']['grading_disagreement'])})
if rows:
    df=pd.DataFrame(rows);df.to_csv(ROOT/'intervention_results.csv',index=False)
    display(df.groupby(['kind','action']).agg(n=('strict_success','size'),accuracy=('strict_success','mean'),passes=('pass_cost','mean')).reset_index())
full=sum(all((pid,a) in by for a in ACTIONS) for pid in CASE_BY_ID)
status={'completed_trials':len(recs),'planned_trials':150,'complete_problem_blocks':full,'planned_blocks':30,
        'study_purpose':'calibration-only mechanism/negative transfer; never refit E06 or claim prospective confirmation',
        'protocol_hash':PROTOCOL_HASH}
(ROOT/'progress.json').write_text(json.dumps(status,indent=2))
print(status);snapshot_export()
